In [1]:
import pandas as pd

In [8]:
df1 = pd.read_parquet("../data/features/zone_stats.parquet")
df2 = pd.read_parquet("../data/features/hour_of_week_baseline.parquet")

In [9]:
df1.head()

,PULocationID,window_end,trip_count,mean_duration
0,1,2024-01-01 11:00:00+00:00,1,0.100000
1,1,2024-01-01 11:15:00+00:00,1,0.100000
2,1,2024-01-01 11:30:00+00:00,1,0.100000
3,1,2024-01-01 11:45:00+00:00,1,0.100000
4,1,2024-01-01 12:00:00+00:00,4,0.145833


In [12]:
df1.describe()

,PULocationID,trip_count,mean_duration
count,962605.000000,962605.000000,852476.000000
mean,141.123497,39.698734,18.846735
std,76.341147,74.059106,19.579642
min,1.000000,1.000000,0.000000
25%,75.000000,1.000000,10.984000
50%,143.000000,4.000000,14.403030
75%,215.000000,42.000000,21.232484
max,265.000000,878.000000,1439.216667


In [5]:
df2.head()

,hour_of_week,window_end,baseline_mean_speed
0,0,2024-02-01 00:00:00+00:00,15.053448
1,1,2024-02-01 00:00:00+00:00,12.055353
2,2,2024-02-01 00:00:00+00:00,10.884630
3,3,2024-02-01 00:00:00+00:00,12.898387
4,4,2024-02-01 00:00:00+00:00,15.304791


In [8]:
df2.window_end.value_counts().sort_index()

window_end
2024-02-01 00:00:00+00:00    168
Name: count, dtype: int64

In [9]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   hour_of_week         168 non-null    int64              
 1   window_end           168 non-null    datetime64[ms, UTC]
 2   baseline_mean_speed  168 non-null    float64            
dtypes: datetime64[ms, UTC](1), float64(1), int64(1)
memory usage: 4.1 KB


In [31]:
import pandas as pd

target_zone = pd.array([22], dtype="int32")[0]
target_hour = pd.array([127], dtype="int32")[0]

# 1. Look up df1 and rename its timestamp
row1 = (
    df1[df1["PULocationID"] == target_zone]
    .sort_values("window_end")
    .rename(columns={"window_end": "zone_window_end"})
    .reset_index(drop=True)
)

# 2. Look up df2 and rename its timestamp
row2 = (
    df2[df2["hour_of_week"] == target_hour]
    .sort_values("window_end")
    .rename(columns={"window_end": "baseline_window_end"})
    .reset_index(drop=True)
)

# 3. Combine both into one row, keeping both distinct timestamps
joined_df = pd.concat([row1, row2], axis=1)

In [41]:
row1.tail()

,PULocationID,zone_window_end,trip_count,mean_duration
1699,22,2024-03-31 12:30:00+00:00,1,NaN
1700,22,2024-03-31 12:45:00+00:00,1,NaN
1701,22,2024-03-31 13:00:00+00:00,1,25.266667
1702,22,2024-03-31 13:15:00+00:00,1,25.266667
1703,22,2024-04-01 01:45:00+00:00,1,NaN


In [27]:
row2.head()

,hour_of_week,baseline_window_end,baseline_mean_speed
0,127,2024-02-01 00:00:00+00:00,21.777862


In [47]:
import pandas as pd

target_zone = pd.array([22], dtype="int32")[0]
target_hour = pd.array([127], dtype="int32")[0]

# Define the exact UTC timestamps to filter
target_timestamps = pd.to_datetime(
    [
        "2024-01-01 06:00:00+00:00",
        "2024-01-01 06:15:00+00:00",
        "2024-01-01 06:30:00+00:00",
        "2024-01-01 06:45:00+00:00",
        "2024-01-01 07:00:00+00:00",
        "2024-01-01 07:15:00+00:00",
        "2024-01-01 07:30:00+00:00",
        "2024-01-01 07:45:00+00:00",
        "2024-01-01 08:00:00+00:00",
    ]
)

# 1. Filter df1 by both PULocationID and the target window_end timestamps
rows1 = (
    df1[
        (df1["PULocationID"] == target_zone)
        & (df1["window_end"].isin(target_timestamps))
    ]
    .sort_values("window_end")
    .rename(columns={"window_end": "zone_window_end"})
    .reset_index(drop=True)
)

# 2. Look up df2 for hour_of_week == 127
row2 = (
    df2[df2["hour_of_week"] == target_hour]
    .sort_values("window_end")
    .tail(1)
    .rename(columns={"window_end": "baseline_window_end"})
    .reset_index(drop=True)
)

# 3. Cross-join (since rows1 has 2 rows and row2 has 1 static baseline row)
joined_df = rows1.merge(row2, how="cross")

In [48]:
rows1

,PULocationID,zone_window_end,trip_count,mean_duration
0,22,2024-01-01 06:45:00+00:00,1,11.766667
1,22,2024-01-01 07:00:00+00:00,1,11.766667
2,22,2024-01-01 07:15:00+00:00,1,11.766667
3,22,2024-01-01 07:30:00+00:00,1,11.766667


In [3]:
data1 = pd.read_parquet(
    "../../mlops-reproducible-pipeline/data/raw/yellow_tripdata_2024-01.parquet"
)
data2 = pd.read_parquet(
    "../../mlops-reproducible-pipeline/data/raw/yellow_tripdata_2024-02.parquet"
)
data3 = pd.read_parquet(
    "../../mlops-reproducible-pipeline/data/raw/yellow_tripdata_2024-03.parquet"
)

In [4]:
data1.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [8]:
data1.size, data2.size, data3.size

(56327856, 57142994, 68069932)